### Practical Components of Geo-Deep-Learning (GDL)

Why we built GDL?

**Because practitioners need a modular, composable, extensible framework to build deep learning models for geospatial tasks.**



The goal: **To provide a clean, unified training framework for small and large geospatial vision models.**

![GDL Framework](/home/valhassa/Projects/nrcan/geo-deep-learning/notebooks/imgs/pytorch_and_lightning.jpg)


At the end of this talk, I hope to have achieved the following:

1. Introduce participants to Geo-Deep-Learning
2. Understand the technical components of GDL
3. How to train and evaluate a model


### Overview of GDL
![GDL Framework](/home/valhassa/Projects/nrcan/geo-deep-learning/notebooks/imgs/gdl_framework.png)

### Component 1: Datasets

Datasets are the main entry point into the whole pipeline.

It loads patches, normalize them, attach metadata, and hand them off to the DataModule.

**Note**: There is a preprocessing that preludes datasets
**Geotiff-Tiler**: https://github.com/valhassan/geotiff-tiler


![Alt text](/home/valhassa/Projects/nrcan/geo-deep-learning/notebooks/imgs/patch_outline.png "Optional title text")


GDL provides two dataset types depending on scale:

| Dataset Class      | Description                                                   | Key Features |
|--------------------|---------------------------------------------------------------|--------------|
| **CSVDataset**     | Simple, local, easy to inspect. Ideal for prototyping or small patch-based datasets. | - Reads image + mask paths from a CSV file |
| **ShardedDataset (WebDataset)** | Scalable, multi-sensor, HPC-ready. Designed for large EO workloads. | - Reads sharded `.tar` WebDataset files<br>- Supports images, labels, metadata<br>- Loads sensor-specific normalization stats from JSON<br>- Automatically distributes shards across GPUs/nodes in DDP |

### Component 2: Datamodules


DataModules are the orchestration layer between Datasets and the Trainer.

Datasets = what the data is


DataModules = how the data flows


GDL provides complementary datamodules to the datasets.


| DataModule               | Purpose                                  | What It Does | Key Features |
|--------------------------|-------------------------------------------|---------------|--------------|
| **CSVDataModule**        | Simple, folder-based loading for small/local experiments | - Instantiates 3 **CSVDatasets** (train/val/test)<br>- Wraps them with standard PyTorch DataLoaders | - Train shuffling<br>- Multi-worker loading<br>- Prefetching, pin-memory, persistent workers |
| **MultiSensorDataModule** | Scalable, multi-sensor, HPC-oriented workloads | - Uses `create_sensor_datasets()` to build **multiple WebDatasets** (one per sensor)<br>- Uses WebLoader for train/val | - Sensor mixing via `RandomMix()`<br>- DDP-aware shard splitting<br>- WebDataset-based batching<br>- Shuffling at shard + sample levels<br>- Optional epoch sizing |



### Component 3: Models

This is where the learning happens. 

This component defines the model into a single, modular unit ready for training. 





#### Overview
![GDL MODEL](/home/valhassa/Projects/nrcan/geo-deep-learning/notebooks/imgs/gdl_model_component.png)

#### Model Structure



| Component   | Purpose                                                | Examples                                                       |
| ----------- | ------------------------------------------------------ | -------------------------------------------------------------- |
| **Encoder** | Extracts features from EO imagery                      | ResNet, MixTransformer (SegFormer), DOFA Encoder, Dinov3 |
| **Neck**    | Optional intermediate fusion or multi-scale processing | MultiLevelNeck (connects vit backbone and decoder_heads.)                        |
| **Decoder** | Converts features into pixel-level predictions         | UNet decoder, SegFormer decoder, UperNet decoder             |


#### Task and Model

![GDL Task](/home/valhassa/Projects/nrcan/geo-deep-learning/notebooks/imgs/gdl_task_with_model.png)

#### Losses, Optimizers, Schedulers


| Component      | Purpose                          | Examples                                 |
| -------------- | -------------------------------- | ---------------------------------------- |
| **Losses**     | Guides learning                  | CrossEntropy, Dice, Focal, hybrid losses |
| **Optimizers** | Updates weights                  | AdamW, SGD                               |
| **Schedulers** | Controls learning rate over time | CosineAnnealing, OneCycle, StepLR        |


#### Design


✔️ **Modular**:  encoder / neck / decoder are swappable


✔️ **Composable**: DOFA, SegFormer, UNet++ share the same interface


✔️ **Configurable**: losses, optimizers, schedulers via YAML


✔️ **Metadata-aware**: temporal, spatial, spectral support for EO


✔️ **Unified**: same interface for training + testing


✔️ **Reproducible**: Lightning + MLflow handle logs & checkpoints


✔️ **Scalable**: multi-GPU, multi-node, multi-sensor ready



### Component 4: Trainer + LightningCLI

The trainer handles the following:

| Functions                 | What it does                            |
| ------------------------- | --------------------------------------- |
| **fit()**                 | Full training loop                      |
| **test()**                | Evaluate best checkpoint after training |
| **accelerator**           | GPU/CPU selection                       |
| **strategy**              | DDP/Multi-GPU logic                     |
| **callbacks**             | Checkpointing, logging, visualization   |
| **loggers**               | MLFlow integration                      |
| **metrics logging**       | During training/val/test                |
| **gradient accumulation** | Efficient multi-GPU                     |
| **precision control**     | Mixed precision (fp16/bf16)             |

**LightningCLI** sits above the Trainer and orchestrates the workflow by wiring together:

- Dataset

- DataModule

- Model

- Trainer configs

- Logging

- Checkpointing


#### Using YAML configuration.

```yaml
trainer:
  max_epochs: 50
  accelerator: gpu
  devices: 4
  precision: bf16
  strategy: ddp

model:
  encoder: mit_b2
  num_classes: 6
  optimizer:
    class_path: torch.optim.AdamW
    init_args:
      lr: 1e-4

data:
  class_path: geo_deep_learning.datamodules.MultiSensorDataModule
  init_args:
    sensor_configs_path: configs/sensors.yaml
    batch_size: 16
```
